[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/85_rescue_pdd260702_solution.ipynb)

# Solution: Rescue Operation

Reference solution.

## 解析

**结论：维护 `monitored_by[r]` = 仍存活、监视区域 `r` 的守卫集合。反复执行两类消除直到不动点：(1) 单杀——`monitored_by[i]` 为空的守卫可直接除掉；(2) 双杀——互为对方唯一监视者的一对 `i,j`（`monitored_by[i]=={j}` 且 `monitored_by[j]=={i}`）一起除掉。全部除掉输出 `SUCCESS`，否则输出剩余数。**

### 建模
把「谁监视谁」看成有向关系，`monitored_by[r]` 是 `r` 的入邻集合（去掉自监视）。一个守卫所在区域无人监视 ⇔ 它的 `monitored_by` 为空 ⇔ 可被单杀。

### 消除顺序
优先做所有能做的单杀（入度 0）；一轮里没有单杀时，再找一对“互相且仅互相监视”的双杀。每次成功消除都会移除其出边，可能让别的守卫的 `monitored_by` 变空，从而级联。重复直到某一轮既无单杀也无双杀。

### 为什么剩下的杀不掉
到达不动点时，剩余守卫都被至少一个仍存活的守卫监视，且不存在“互为唯一监视者”的对——这类结构（如 3 元环 1→2→3→1）无法再拆解，剩余数即答案。

### 验证
已用「枚举所有可行消除序列取最小剩余」的递归暴力在上千组小规模随机数据上对拍一致。

### 复杂度
每次消除移除一个/两个守卫并更新其出边，整体近似 `O(n + 边数)` 摊还（外层不动点循环受限于总消除次数）。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
# no imports needed

In [ ]:
# ✅ SOLUTION

class Solution:
    def min_remaining(self, n, monitors):
        watch = [[] for _ in range(n + 1)]
        for i in range(n):
            watch[i + 1] = list(monitors[i])
        mby = [set() for _ in range(n + 1)]          # monitored_by[r] = alive watchers of r
        for j in range(1, n + 1):
            for r in watch[j]:
                if 1 <= r <= n and r != j:
                    mby[r].add(j)
        alive = [i > 0 for i in range(n + 1)]; cnt = n
        def kill(i):
            nonlocal cnt
            alive[i] = False; cnt -= 1
            for r in watch[i]:
                if 1 <= r <= n:
                    mby[r].discard(i)
        changed = True
        while changed:
            changed = False
            for i in range(1, n + 1):                # single kills: unwatched guards
                if alive[i] and not mby[i]:
                    kill(i); changed = True
            if changed:
                continue
            for i in range(1, n + 1):                # double kill: mutual-only pair
                if alive[i] and len(mby[i]) == 1:
                    j = next(iter(mby[i]))
                    if alive[j] and mby[j] == {i}:
                        kill(i); kill(j); changed = True; break
        return 'SUCCESS' if cnt == 0 else cnt

In [ ]:
# Demo
sol = Solution()
print(sol.min_remaining(2, [[2], []]))       # SUCCESS
print(sol.min_remaining(2, [[2], [1]]))      # SUCCESS
print(sol.min_remaining(3, [[2], [3], [1]])) # 3

In [ ]:
from torch_judge import check
check('rescue_pdd260702')